# Treasury Multi-Curve: Bond Curve vs SOFR vs Repo

**The real-world setup:** a Treasury desk uses three curves simultaneously.

1. **Treasury curve** — bootstrapped from on-the-run bond prices (what the bond market says)
2. **SOFR/OIS curve** — bootstrapped from swap rates (what the derivatives market says)
3. **Repo curve** — from repo market quotes (what financing costs)

The spreads between these curves are where trading opportunities live.

Market data modelled on **15 Jul 2024** — realistic levels for an inverted US yield curve environment.

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.bond import FixedRateBond
from pricebook.bootstrap import bootstrap
from pricebook.bond_desk import fit_curve_from_bonds
from pricebook.bond_trading_desk import bond_risk_metrics, bond_carry_roll
from pricebook.repo_curve import build_repo_curve, repo_carry_from_curve
from pricebook.treasury_quoting import to_32nds
from pricebook.viz import configure_theme, greeks_profile
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 7, 15)
print(f"Valuation date: {REF}\n")

## 1. Build the three curves from market data

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CURVE 1: Treasury curve — from on-the-run bond market prices
# ═══════════════════════════════════════════════════════════════

# On-the-run Treasury notes: coupon, issue date, maturity, market clean price
# Prices reflect Jul 2024 inverted curve (short yields > long yields)
ust_bonds = [
    ("2Y",  0.0475, date(2024, 6, 30), date(2026, 6, 30),  99.750),
    ("3Y",  0.0450, date(2024, 5, 15), date(2027, 5, 15),  99.375),
    ("5Y",  0.0425, date(2024, 4, 30), date(2029, 4, 30),  99.000),
    ("7Y",  0.0400, date(2024, 3, 31), date(2031, 3, 31),  97.500),
    ("10Y", 0.0425, date(2024, 2, 15), date(2034, 2, 15), 102.500),
    ("20Y", 0.0450, date(2024, 2, 15), date(2044, 2, 15),  99.250),
    ("30Y", 0.0437, date(2024, 2, 15), date(2054, 2, 15),  96.000),
]

bonds_and_prices = []
for label, coupon, issue, mat, clean_px in ust_bonds:
    bond = FixedRateBond.treasury_note(issue, mat, coupon)
    bonds_and_prices.append((bond, clean_px))

treasury_curve, fitted_bonds = fit_curve_from_bonds(REF, bonds_and_prices)

print("CURVE 1: Treasury Discount Curve (from bond prices)")
print(f"{'Tenor':>5}  {'Coupon':>7}  {'MktPrice':>9}  {'FitPrice':>9}  {'Resid':>7}  {'Zero':>7}")
print(f"{'─'*5}  {'─'*7}  {'─'*9}  {'─'*9}  {'─'*7}  {'─'*7}")
for (label, cpn, issue, mat, mkt_px), fb in zip(ust_bonds, fitted_bonds):
    T = (mat - REF).days / 365.25
    zero = treasury_curve.zero_rate(mat)
    print(f"{label:>5}  {cpn*100:>6.2f}%  {mkt_px:>9.3f}  {fb.fitted_price:>9.3f}  {fb.residual:>+6.3f}  {zero*100:>6.2f}%")

# ═══════════════════════════════════════════════════════════════
# CURVE 2: SOFR/OIS curve — from swap market
# ═══════════════════════════════════════════════════════════════

deposits = [
    (REF + relativedelta(months=1), 0.0533),
    (REF + relativedelta(months=3), 0.0531),
    (REF + relativedelta(months=6), 0.0518),
]
swaps = [
    (REF + relativedelta(years=1), 0.0495),
    (REF + relativedelta(years=2), 0.0460),
    (REF + relativedelta(years=3), 0.0440),
    (REF + relativedelta(years=5), 0.0415),
    (REF + relativedelta(years=7), 0.0405),
    (REF + relativedelta(years=10), 0.0395),
    (REF + relativedelta(years=20), 0.0410),
    (REF + relativedelta(years=30), 0.0420),
]
sofr_curve = bootstrap(REF, deposits, swaps)
print("\nCURVE 2: SOFR/OIS Curve (from swaps) — built")

# ═══════════════════════════════════════════════════════════════
# CURVE 3: Repo curve — from repo market
# ═══════════════════════════════════════════════════════════════

repo_curve = build_repo_curve(REF, {
    "ON": 0.0532, "1W": 0.0530, "1M": 0.0528,
    "3M": 0.0523, "6M": 0.0515, "1Y": 0.0495,
})
print("CURVE 3: GC Repo Curve (from repo market) — built")

## 2. Compare the three curves — where are the spreads?

In [ ]:
# Zero rates from all three curves at standard tenors
tenors = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
tsy_zeros, sofr_zeros, repo_zeros = [], [], []

for t in tenors:
    d = REF + relativedelta(months=int(t * 12)) if t < 1 else REF + relativedelta(years=int(t))
    tsy_zeros.append(treasury_curve.zero_rate(d) * 100)
    sofr_zeros.append(sofr_curve.zero_rate(d) * 100)
    repo_zeros.append(repo_curve.rate_at(int(t * 365)) * 100)

# Plot all three curves
with apply_theme():
    fig, (ax1, ax2) = create_figure(2)

    ax1.plot(tenors, tsy_zeros, 'o-', linewidth=2, label="Treasury (bond-fitted)")
    ax1.plot(tenors, sofr_zeros, 's-', linewidth=2, label="SOFR/OIS (swaps)")
    ax1.plot(tenors[:6], repo_zeros[:6], '^-', linewidth=2, label="Repo (GC)")
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("Zero Rate (%)")
    ax1.set_title("Three Curves: Treasury vs SOFR vs Repo")
    ax1.legend(fontsize=9)

    # Treasury-OIS spread (the basis trade signal)
    tsy_ois_spread = [t - s for t, s in zip(tsy_zeros, sofr_zeros)]
    ax2.bar([f"{t}" for t in tenors], tsy_ois_spread, alpha=0.8)
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.set_xlabel("Tenor")
    ax2.set_ylabel("Treasury - SOFR (bp × 100)")
    ax2.set_title("Treasury-OIS Spread (negative = Treasuries rich vs swaps)")

    fig.tight_layout()

# Table
print(f"\n{'Tenor':>6}  {'Treasury':>9}  {'SOFR':>9}  {'Repo':>9}  {'TSY-SOFR':>9}")
print(f"{'─'*6}  {'─'*9}  {'─'*9}  {'─'*9}  {'─'*9}")
for i, t in enumerate(tenors):
    repo_str = f"{repo_zeros[i]:.2f}%" if i < 6 else "—"
    spread = tsy_zeros[i] - sofr_zeros[i]
    print(f"{t:>5}Y  {tsy_zeros[i]:>8.2f}%  {sofr_zeros[i]:>8.2f}%  {repo_str:>9}  {spread*100:>+7.1f}bp")

## 3. Price the same bond off each curve — what's the difference?

In [ ]:
# Take the 10Y on-the-run note
bond_10y = bonds_and_prices[4][0]  # 4.25% Feb 2034
mkt_clean = ust_bonds[4][4]        # 102.500
settle = bond_10y.settlement_date(REF)
ai = bond_10y.accrued_interest(settle)

# Price off each curve
price_tsy = bond_10y.dirty_price(treasury_curve)
price_sofr = bond_10y.dirty_price(sofr_curve)
ytm_tsy = bond_10y.yield_to_maturity(price_tsy, settle)
ytm_sofr = bond_10y.yield_to_maturity(price_sofr, settle)
ytm_mkt = bond_10y.yield_to_maturity(mkt_clean + ai, settle)

print("10Y T-Note: 4.25% due Feb 2034")
print(f"  Market clean price: {mkt_clean:.3f} ({to_32nds(mkt_clean)})")
print(f"  Accrued interest:   {ai:.4f}")
print()
print(f"{'Curve':<20}  {'Dirty Price':>11}  {'Clean':>11}  {'YTM':>8}  {'vs Market':>10}")
print(f"{'─'*20}  {'─'*11}  {'─'*11}  {'─'*8}  {'─'*10}")
print(f"{'Market':<20}  {mkt_clean + ai:>11.4f}  {mkt_clean:>11.3f}  {ytm_mkt*100:>7.3f}%  {'—':>10}")
print(f"{'Treasury curve':<20}  {price_tsy:>11.4f}  {price_tsy - ai:>11.3f}  {ytm_tsy*100:>7.3f}%  {(price_tsy - mkt_clean - ai) * 100:>+8.1f}ct")
print(f"{'SOFR/OIS curve':<20}  {price_sofr:>11.4f}  {price_sofr - ai:>11.3f}  {ytm_sofr*100:>7.3f}%  {(price_sofr - mkt_clean - ai) * 100:>+8.1f}ct")

print(f"\nTreasury-SOFR price difference: {(price_tsy - price_sofr) * 100:+.1f} cents")
print(f"Treasury-SOFR yield spread:     {(ytm_tsy - ytm_sofr) * 1e4:+.1f}bp")
print(f"\nInterpretation:")
if price_tsy > price_sofr:
    print(f"  Treasury curve gives HIGHER price → Treasuries are RICH vs SOFR")
    print(f"  Trade: pay fixed in swaps + buy Treasuries (basis tightener)")
else:
    print(f"  Treasury curve gives LOWER price → Treasuries are CHEAP vs SOFR")
    print(f"  Trade: receive fixed in swaps + sell Treasuries (basis widener)")

## 4. Carry analysis — what does it cost to hold the position?

In [ ]:
# Carry from each financing source
face = 10_000_000  # $10M position

# 1. Carry using repo curve (term-matched)
repo_carry = repo_carry_from_curve(price_tsy, bond_10y.coupon_rate, repo_curve, hold_days=30)

# 2. Carry using flat SOFR rate
sofr_1m = sofr_curve.zero_rate(REF + relativedelta(months=1))

# 3. Carry using bond_carry_roll (from bond desk)
desk_carry = bond_carry_roll(bond_10y, treasury_curve, repo_rate=repo_carry.repo_rate_used,
                              horizon_days=30, settlement=settle)

print("10Y T-Note Carry Analysis ($10M, 30-day horizon)")
print(f"{'─'*60}")
print(f"  Bond coupon:       {bond_10y.coupon_rate*100:.2f}% → ${bond_10y.coupon_rate * face * 30 / 365:>+10,.0f}")
print(f"  Repo rate (1M):    {repo_carry.repo_rate_used*100:.2f}% → ${repo_carry.repo_cost * face / 100:>+10,.0f}")
print(f"  SOFR rate (1M):    {sofr_1m*100:.2f}%")
print(f"  Net carry (repo):  ${repo_carry.net_carry * face / 100:>+10,.0f}")
print(f"  Annualised:        {repo_carry.annualised_carry_bps:+.0f}bp")
print(f"{'─'*60}")

# Carry at different repo tenors
print(f"\nCarry by repo tenor:")
print(f"{'Tenor':>6}  {'Repo Rate':>9}  {'Net Carry':>12}  {'Ann (bp)':>9}")
print(f"{'─'*6}  {'─'*9}  {'─'*12}  {'─'*9}")
for label, days in [("O/N", 1), ("1W", 7), ("1M", 30), ("3M", 91)]:
    rc = repo_carry_from_curve(price_tsy, bond_10y.coupon_rate, repo_curve, hold_days=days)
    print(f"{label:>6}  {rc.repo_rate_used*100:>8.2f}%  ${rc.net_carry * face / 100:>+11,.0f}  {rc.annualised_carry_bps:>+8.0f}bp")

print(f"\nInverted curve environment: short-term repo ({repo_carry.repo_rate_used*100:.0f}bp) > bond coupon ({bond_10y.coupon_rate*100:.0f}bp)")
print(f"→ NEGATIVE carry: you PAY to hold this bond. Roll-down must compensate.")

## Summary

| Curve | Source | What it tells you |
|-------|--------|-------------------|
| **Treasury** | On-the-run bond prices | Fair value in the bond market |
| **SOFR/OIS** | Swap rates | Fair value in the derivatives market |
| **Repo** | Repo market | Financing cost for holding bonds |

| Spread | Signal |
|--------|--------|
| **Treasury - SOFR** | Basis trade: are Treasuries rich or cheap vs swaps? |
| **Coupon - Repo** | Carry: do you earn or pay to hold? |
| **Roll-down** | Curve steepness: does aging help? |

In an inverted curve (Jul 2024): negative carry everywhere, but long-end roll-down partially compensates. The basis (Treasury vs SOFR) drives relative value between cash and derivatives.